In [1]:
import sys

print("=" * 70)
print("FINANCIAL NEWS SENTIMENT ANALYSIS")
print("=" * 70)
print("Python version:", sys.version)

StatementMeta(, 234d189a-79c1-4815-ae00-e6c113b582d2, 3, Finished, Available, Finished, False)

FINANCIAL NEWS SENTIMENT ANALYSIS
Python version: 3.11.8 (main, Feb 26 2024, 21:39:34) [GCC 11.2.0]


In [2]:
try:
    import transformers
    import torch

    print("transformers version:", transformers.__version__)
    print("torch version:", torch.__version__)
    print("AI libraries are available.")

except ImportError as e:
    print("Missing library:", e)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 4, Finished, Available, Finished, False)

transformers version: 4.37.2
torch version: 2.2.1
AI libraries are available.


In [3]:
from pyspark.sql import functions as F
import pandas as pd

clean_news_spark = spark.table("financial_news_clean")

print("Articles available:", clean_news_spark.count())

display(
    clean_news_spark.select(
        "Article_ID",
        "Ticker",
        "Published_At",
        "Headline",
        "Summary"
    ).limit(10)
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 5, Finished, Available, Finished, False)

Articles available: 1853


SynapseWidget(Synapse.DataFrame, 9bee3705-dbe7-4600-b387-b38ee7bf2958)

In [4]:
sentiment_input_df = (
    clean_news_spark
    .withColumn(
        "Sentiment_Text",
        F.concat_ws(
            ". ",
            F.coalesce(F.col("Headline"), F.lit("")),
            F.coalesce(F.col("Summary"), F.lit(""))
        )
    )
    .withColumn(
        "Sentiment_Text",
        F.trim(F.col("Sentiment_Text"))
    )
)

display(
    sentiment_input_df.select(
        "Article_ID",
        "Ticker",
        "Headline",
        "Sentiment_Text"
    ).limit(10)
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf86a8a3-2c80-412f-a6de-82c5976c5bf7)

In [5]:
test_sentiment_pdf = (
    sentiment_input_df
    .select(
        "Article_ID",
        "Ticker",
        "Headline",
        "Sentiment_Text"
    )
    .limit(20)
    .toPandas()
)

print("Test articles:", len(test_sentiment_pdf))

display(test_sentiment_pdf.head())

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 7, Finished, Available, Finished, False)

Test articles: 20


SynapseWidget(Synapse.DataFrame, 276bd406-38a3-4928-8469-a150a79ec95e)

In [6]:
from transformers import pipeline

MODEL_NAME = "ProsusAI/finbert"

sentiment_model = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    return_all_scores=True
)

print("FinBERT loaded successfully.")

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 8, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 25, Finished, Available, Finished, False)

In [7]:
sample_text = (
    "NVIDIA reported strong revenue growth driven by increasing demand "
    "for artificial intelligence chips."
)

sample_result = sentiment_model(sample_text)

print(sample_result)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 9, Finished, Available, Finished, False)

[[{'label': 'positive', 'score': 0.9551683068275452}, {'label': 'negative', 'score': 0.01536376029253006}, {'label': 'neutral', 'score': 0.029467923566699028}]]


In [8]:
def analyze_sentiment(text):
    """
    Analyze financial text with FinBERT.

    Returns:
        Sentiment
        Confidence
        Sentiment_Score
    """

    if text is None or not str(text).strip():
        return "neutral", 0.0, 0.0

    result = sentiment_model(
        str(text)[:2000],
        truncation=True
    )[0]

    scores = {
        item["label"].lower(): float(item["score"])
        for item in result
    }

    sentiment = max(scores, key=scores.get)
    confidence = scores[sentiment]

    sentiment_score = (
        scores.get("positive", 0.0)
        - scores.get("negative", 0.0)
    )

    return sentiment, confidence, sentiment_score

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 10, Finished, Available, Finished, False)

In [9]:
test_function_result = analyze_sentiment(
    "NVIDIA reported strong growth and higher demand for AI chips."
)

print(test_function_result)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 11, Finished, Available, Finished, False)

('positive', 0.9581623673439026, 0.9426241796463728)


In [10]:
sentiment_results = test_sentiment_pdf["Sentiment_Text"].apply(
    analyze_sentiment
)

test_sentiment_pdf[
    [
        "Sentiment",
        "Confidence",
        "Sentiment_Score"
    ]
] = pd.DataFrame(
    sentiment_results.tolist(),
    index=test_sentiment_pdf.index
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 12, Finished, Available, Finished, False)

In [11]:
display(
    test_sentiment_pdf[
        [
            "Ticker",
            "Headline",
            "Sentiment",
            "Confidence",
            "Sentiment_Score"
        ]
    ]
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2bccecca-00a5-4f49-bec6-21740230fe06)

In [12]:
print("Sentiment distribution:")
print(test_sentiment_pdf["Sentiment"].value_counts())

print()
print("Average confidence:")
print(round(test_sentiment_pdf["Confidence"].mean(), 4))

print()
print("Average sentiment score:")
print(round(test_sentiment_pdf["Sentiment_Score"].mean(), 4))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 14, Finished, Available, Finished, False)

Sentiment distribution:
Sentiment
neutral     9
negative    6
positive    5
Name: count, dtype: int64

Average confidence:
0.8456

Average sentiment score:
-0.0793


In [13]:
all_sentiment_pdf = (
    sentiment_input_df
    .select(
        "Article_ID",
        "Sentiment_Text"
    )
    .toPandas()
)

all_sentiment_pdf["Sentiment_Text"] = (
    all_sentiment_pdf["Sentiment_Text"]
    .fillna("")
    .astype(str)
)

print("Articles ready for sentiment analysis:", len(all_sentiment_pdf))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 15, Finished, Available, Finished, False)

Articles ready for sentiment analysis: 1853


In [14]:
texts = all_sentiment_pdf["Sentiment_Text"].tolist()

all_predictions = sentiment_model(
    texts,
    batch_size=16,
    truncation=True,
    max_length=512
)

print("Predictions completed:", len(all_predictions))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 16, Finished, Available, Finished, False)

Predictions completed: 1853


In [15]:
def parse_finbert_prediction(prediction):
    scores = {
        item["label"].lower(): float(item["score"])
        for item in prediction
    }

    sentiment = max(scores, key=scores.get)
    confidence = scores[sentiment]

    positive_score = scores.get("positive", 0.0)
    negative_score = scores.get("negative", 0.0)
    neutral_score = scores.get("neutral", 0.0)

    sentiment_score = positive_score - negative_score

    return {
        "Sentiment": sentiment,
        "Confidence": confidence,
        "Positive_Score": positive_score,
        "Negative_Score": negative_score,
        "Neutral_Score": neutral_score,
        "Sentiment_Score": sentiment_score
    }


parsed_predictions = [
    parse_finbert_prediction(prediction)
    for prediction in all_predictions
]

prediction_pdf = pd.DataFrame(parsed_predictions)

display(prediction_pdf.head(10))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7df6b694-0b7e-4454-a638-d7c6570def01)

In [16]:
sentiment_results_pdf = pd.concat(
    [
        all_sentiment_pdf[["Article_ID"]].reset_index(drop=True),
        prediction_pdf.reset_index(drop=True)
    ],
    axis=1
)

print("Sentiment result rows:", len(sentiment_results_pdf))

display(sentiment_results_pdf.head(10))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 18, Finished, Available, Finished, False)

Sentiment result rows: 1853


SynapseWidget(Synapse.DataFrame, 8ac0a5b3-773c-491a-88af-ba21b4e683b6)

In [17]:
print("Sentiment distribution:")
print(sentiment_results_pdf["Sentiment"].value_counts())

print()
print("Average confidence:")
print(round(sentiment_results_pdf["Confidence"].mean(), 4))

print()
print("Average sentiment score:")
print(round(sentiment_results_pdf["Sentiment_Score"].mean(), 4))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 19, Finished, Available, Finished, False)

Sentiment distribution:
Sentiment
neutral     684
negative    587
positive    582
Name: count, dtype: int64

Average confidence:
0.821

Average sentiment score:
0.0023


In [18]:
sentiment_scores_spark = spark.createDataFrame(
    sentiment_results_pdf
)

print("Spark sentiment rows:", sentiment_scores_spark.count())

display(sentiment_scores_spark.limit(10))

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 20, Finished, Available, Finished, False)

Spark sentiment rows: 1853


SynapseWidget(Synapse.DataFrame, 16c0ad9d-044b-49d8-a6f0-3fbcd625cae9)

In [19]:
from pyspark.sql import functions as F

financial_news_sentiment_df = (
    clean_news_spark
    .join(
        sentiment_scores_spark,
        on="Article_ID",
        how="left"
    )
    .withColumn(
        "Bullish_Bearish",
        F.when(F.col("Sentiment") == "positive", F.lit("Bullish"))
         .when(F.col("Sentiment") == "negative", F.lit("Bearish"))
         .otherwise(F.lit("Neutral"))
    )
    .withColumn(
        "Sentiment_Model",
        F.lit("ProsusAI/finbert")
    )
    .withColumn(
        "Scored_At",
        F.current_timestamp()
    )
)

print("Final sentiment rows:", financial_news_sentiment_df.count())

display(
    financial_news_sentiment_df.select(
        "Article_ID",
        "Ticker",
        "Headline",
        "Sentiment",
        "Confidence",
        "Sentiment_Score",
        "Bullish_Bearish"
    ).limit(20)
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 21, Finished, Available, Finished, False)

Final sentiment rows: 1853


SynapseWidget(Synapse.DataFrame, 1a4b99bd-2479-41d4-ba07-7cc2aa283694)

In [20]:
(
    financial_news_sentiment_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("financial_news_sentiment")
)

print("Table financial_news_sentiment saved successfully.")

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 22, Finished, Available, Finished, False)

Table financial_news_sentiment saved successfully.


In [21]:
display(
    spark.sql("""
        SELECT
            Sentiment,
            Bullish_Bearish,
            COUNT(*) AS Article_Count,
            ROUND(AVG(Confidence), 4) AS Average_Confidence,
            ROUND(AVG(Sentiment_Score), 4) AS Average_Sentiment_Score
        FROM financial_news_sentiment
        GROUP BY Sentiment, Bullish_Bearish
        ORDER BY Article_Count DESC
    """)
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2bdb71b2-085d-4f08-ab8c-bcc448b1e6e4)

In [22]:
display(
    spark.sql("""
        SELECT
            Ticker,
            COUNT(*) AS Articles_Analyzed,
            SUM(CASE WHEN Sentiment = 'positive' THEN 1 ELSE 0 END)
                AS Positive_Articles,
            SUM(CASE WHEN Sentiment = 'neutral' THEN 1 ELSE 0 END)
                AS Neutral_Articles,
            SUM(CASE WHEN Sentiment = 'negative' THEN 1 ELSE 0 END)
                AS Negative_Articles,
            ROUND(AVG(Sentiment_Score), 4)
                AS Average_Sentiment_Score,
            ROUND(AVG(Confidence), 4)
                AS Average_Confidence
        FROM financial_news_sentiment
        GROUP BY Ticker
        ORDER BY Average_Sentiment_Score DESC
    """)
)

StatementMeta(, 96d3d968-32a4-4195-be5b-182647fb38f4, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6eeac96-d628-4a31-8c7d-d25b6200b09b)